# Murmur Mamba3 MIMO — RTX 6000 Ada 40M smoke

A self-contained, from-scratch compatibility and stability test. It stops immediately if the official Mamba3 MIMO forward/backward gate fails. Passing this notebook is required before a 350M MIMO session.


In [ ]:
# 1. Verify the rented Linux GPU and clone this exact experiment branch
from pathlib import Path
import os, subprocess, sys
import torch
EXPECTED_GPU = 'RTX 6000 Ada'
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Start a Linux instance with an RTX 6000 Ada GPU.')
gpu_name = torch.cuda.get_device_name(0)
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu_name})
if EXPECTED_GPU.lower() not in gpu_name.lower():
    raise RuntimeError(f'This notebook is pinned to {EXPECTED_GPU}; found {gpu_name}. Stop and use the matching notebook.')
!nvidia-smi
repo_dir = Path('/workspace/murmur-science')
branch = 'codex/rtx6000-ada-mimo-smoke'
if repo_dir.exists():
    %cd /workspace/murmur-science
    subprocess.check_call(['git', 'fetch', 'origin', branch])
    subprocess.check_call(['git', 'checkout', branch])
    subprocess.check_call(['git', 'pull', '--ff-only', 'origin', branch])
else:
    subprocess.check_call(['git', 'clone', '--branch', branch, '--single-branch', 'https://github.com/orkrs/murmur-science.git', str(repo_dir)])
    %cd /workspace/murmur-science
%pip install -q --no-deps -e .
sys.path[:0] = ['/workspace/murmur-science/src', '/workspace/murmur-science']
Path('artifacts').mkdir(exist_ok=True)
print('Project environment ready')

In [ ]:
# 2. Install the official Mamba3 MIMO stack without replacing the CUDA PyTorch wheel
%pip install -q --upgrade datasets sentencepiece pandas pyarrow einops ninja
%pip install -q --upgrade 'tilelang==0.1.8' 'apache-tvm-ffi<=0.1.12' 'quack-kernels>=0.3.4' 'triton>=3.5.0'
os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
%pip install -q --no-cache-dir --no-deps --force-reinstall --no-build-isolation git+https://github.com/state-spaces/mamba.git@main
import importlib.metadata as md
for package in ('mamba-ssm', 'tilelang', 'apache-tvm-ffi', 'quack-kernels', 'triton'):
    try: print(package, md.version(package))
    except md.PackageNotFoundError: print(package, 'not found')

In [ ]:
# 3. Mandatory RTX 6000 Ada MIMO gate — realistic 40M-width forward + backward
import json, time
from mamba_ssm.modules.mamba3 import Mamba3
gate_path = Path('artifacts/mamba3_mimo_gate.json')
try:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    gate = Mamba3(d_model=512, d_state=64, headdim=64, is_mimo=True, mimo_rank=2, chunk_size=32, dtype=torch.bfloat16).cuda().train()
    x = torch.randn(1, 512, 512, device='cuda', dtype=torch.bfloat16, requires_grad=True)
    torch.cuda.synchronize(); started = time.perf_counter()
    y = gate(x); loss = y.float().square().mean(); loss.backward()
    torch.cuda.synchronize(); elapsed = time.perf_counter() - started
    if not torch.isfinite(y).all() or not torch.isfinite(x.grad).all():
        raise RuntimeError('MIMO forward/backward returned non-finite tensors')
    gate_result = {'passed': True, 'gpu': gpu_name, 'dtype': 'bfloat16', 'shape': list(y.shape), 'seconds': elapsed, 'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 2**30, 3)}
except Exception as exc:
    gate_result = {'passed': False, 'gpu': gpu_name, 'error': repr(exc)}
gate_path.write_text(json.dumps(gate_result, indent=2), encoding='utf-8')
print(gate_result)
if not gate_result['passed']:
    raise RuntimeError('Mamba3 MIMO gate failed. Stop here and send artifacts/mamba3_mimo_gate.json plus this cell output.')

In [ ]:
# 4. Confirm the exact 40M-class MIMO configuration
!python scripts/param_count.py --config configs/rtx6000_ada_mimo_40m_smoke.toml
from murmur.config import load_run_config
config = load_run_config(Path('configs/rtx6000_ada_mimo_40m_smoke.toml'))
assert config.model.mixer == 'mamba3_mimo'
assert config.train.max_tokens == 2_000_000
print('Verified: 40M-class recurrent Mamba3 MIMO (rank 2), 2M-token smoke.')

In [ ]:
# 5. Download a public smoke corpus automatically and write deterministic splits
import hashlib, json, re
from datasets import load_dataset
SOURCE = 'roneneldan/TinyStories'
out = Path('artifacts/rtx6000_mimo_corpus'); out.mkdir(parents=True, exist_ok=True)
budget, accepted, seen, train, val = 2_400_000, 0, set(), [], []
def add(text):
    global accepted
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) < 80: return
    digest = hashlib.sha256(text.encode()).hexdigest(); estimate = max(1, len(text.encode()) // 4)
    if digest in seen or accepted + estimate > budget: return
    seen.add(digest); accepted += estimate
    row = {'text': text, 'source': SOURCE, 'text_sha256': digest}
    (val if int(digest[:8], 16) % 100 < 2 else train).append(row)
for row in load_dataset(SOURCE, split='train', streaming=True):
    add(row['text'])
    if accepted >= budget: break
for name, rows in (('train.jsonl', train), ('val.jsonl', val)):
    with (out / name).open('w', encoding='utf-8') as handle:
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False) + '\n')
with (out / 'corpus.txt').open('w', encoding='utf-8') as handle:
    for row in train + val: handle.write(row['text'] + '\n\n')
if not train or not val: raise RuntimeError('Automatic corpus build produced an empty split')
print({'source': SOURCE, 'train_docs': len(train), 'val_docs': len(val), 'estimated_tokens': accepted})

In [ ]:
# 6. Train a fresh tokenizer and pack token shards
!python scripts/train_tokenizer.py --corpus artifacts/rtx6000_mimo_corpus/corpus.txt --output artifacts/rtx6000_mimo_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/rtx6000_ada_mimo_40m_smoke.toml --tokenizer artifacts/rtx6000_mimo_tokenizer.model --train-input artifacts/rtx6000_mimo_corpus/train.jsonl --val-input artifacts/rtx6000_mimo_corpus/val.jsonl --output artifacts/rtx6000_mimo_data
assert list(Path('artifacts/rtx6000_mimo_data').glob('train_*.bin'))
assert list(Path('artifacts/rtx6000_mimo_data').glob('val_*.bin'))
print('Packed smoke data ready')

In [ ]:
# 7. Train from random weights for 2M tokens
template = Path('configs/rtx6000_ada_mimo_40m_smoke.toml').read_text(encoding='utf-8')
session = template.replace('artifacts/data/train.bin', 'artifacts/rtx6000_mimo_data/train.bin').replace('artifacts/data/val.bin', 'artifacts/rtx6000_mimo_data/val.bin')
Path('configs/rtx6000_ada_mimo_40m_smoke_session.toml').write_text(session, encoding='utf-8')
run_dir = Path('artifacts/runs/rtx6000_ada_mimo_40m_smoke')
subprocess.run(['python', 'scripts/train.py', '--config', 'configs/rtx6000_ada_mimo_40m_smoke_session.toml', '--run-dir', str(run_dir), '--device', 'cuda'], check=True)
assert (run_dir / 'checkpoints' / 'last' / 'COMPLETED').exists()
print('MIMO smoke checkpoint ready')

In [ ]:
# 8. Evaluate and write one hand-off report; cached generation remains intentionally disabled
eval_path = Path('artifacts/eval_rtx6000_ada_mimo_40m.json')
subprocess.run(['python', 'scripts/evaluate.py', '--config', 'configs/rtx6000_ada_mimo_40m_smoke_session.toml', '--checkpoint', 'artifacts/runs/rtx6000_ada_mimo_40m_smoke/checkpoints/last', '--output', str(eval_path), '--device', 'cuda'], check=True)
report = {'gate': json.loads(gate_path.read_text(encoding='utf-8')), 'evaluation': json.loads(eval_path.read_text(encoding='utf-8')), 'checkpoint_complete': (Path('artifacts/runs/rtx6000_ada_mimo_40m_smoke/checkpoints/last/COMPLETED').exists()), 'next_step': '350M MIMO stability run only after review of this report'}
Path('artifacts/mimo_smoke_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print('Send artifacts/mimo_smoke_report.json and the training log for a go/no-go decision on 350M.')